In [ ]:
# Resolve paths consistently from either the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Matching city names

In [ ]:
#sets the notebook width
from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import numpy as np
from pathlib import Path


%matplotlib inline
# figures
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker
import seaborn as sns


def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {'text.usetex': usetex,
              'font.family': 'sans-serif',
              'font.sans-serif': ['Helvetica'],
             # 'text.latex.unicode': True,
              #'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
             #                         r"\usepackage{lmodern}",
            #                          r"\usepackage{amsmath}",
             #                         r"\usepackage{mathptmx}"
             #                         ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction": "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=False)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

# Load the UCR data



In [ ]:
def create_path(year):
    filepath=Path('../data/clean_crime_data/{}.csv'.format(year))
    return filepath

In [ ]:
dfset = []
for year in range(1960,2015):
    fname=create_path(year)
    tdf = pd.read_csv(fname, index_col = 0, low_memory = False)
    tdf['Year'] = int(fname.split('/')[-1].split('.csv')[0])
    dfset.append(tdf)
ucrdf = pd.concat(dfset)
# ucrdf.rename(columns = {"Core City Indication": "msa"}, inplace=True)
ucrdf.MSA = ucrdf.MSA.astype(object)
ucrdf.rename(columns = {"MSA": "ucrfips"}, inplace=True)
ucrdf.rename(columns = {"State": "fipst"}, inplace=True)

In [ ]:
ucrdf[ucrdf.columns[:26]].head().T

In [ ]:
ucrdf['First Line of Mailing Address']=ucrdf['First Line of Mailing Address'].astype(str)
ucrdf['Second Line of Mailing Address']=ucrdf['Second Line of Mailing Address'].astype(str)
ucrdf['Third Line of Mailing Address']=ucrdf['Third Line of Mailing Address'].astype(str)
ucrdf['Fourth Line of Mailing Address']=ucrdf['Fourth Line of Mailing Address'].astype(str)

In [ ]:
ucrdf['NAME']=(ucrdf['First Line of Mailing Address'].str.replace('nan','').str.strip()
               +' '+ ucrdf['Second Line of Mailing Address'].str.replace('nan','').str.strip()
               +' '+ ucrdf['Third Line of Mailing Address'].str.replace('nan','').str.strip()
               +' '+ ucrdf['Fourth Line of Mailing Address'].str.replace('nan','').str.strip())


In [ ]:
ucrdf['NAME']=ucrdf['NAME'].str.strip()

In [ ]:
df=ucrdf[[ 'ORI', 'Group', 'Division', 'Year', 'Sequence Number',
       'Juvenile Age', 'Core City Indication', 'Covered By',
       'Covered By Group', 'Last Update', 'Field Office',
       'Number of Months Reported', 'Agency Count', 'Population', 'County', 'Group I', 'Group II', 'Agency Name', 'Agency State Name',
       'NAME',
       'Zip Code','ucrfips','fipst']]
df.head()

In [ ]:
df=df[df['ORI'].apply(type)==str]

In [ ]:
df[df['ORI'].str.contains("IL")].head().T

In [ ]:
months=['january', 'february', 'march', 'april', 'may', 'june','july', 'august', 'september', 'october', 'november', 'december']

crimes=['Murder',
      'Manslaughter',
      'Rape Total',
      'Robbery Total',
      'Assault Total',
      'Burglary Total',
      'Larceny Total',
      'Motor Vehicle Theft Total']

for crime in crimes:
    print(crime.strip('Total'))

crime_labels=dict()
allcrimes=[]
for crime in crimes:
    print(crime.strip('Total').strip(' '))
    columns=[]
    for month in months:
        columns.append(month+'_'+crime)
        allcrimes.append(month+'_'+crime)
    crime_labels[crime.strip('Total').strip(' ')]=columns

In [ ]:
#ucrdf=ucrdf[ucrdf['ORI'].apply(type)==str]
ucrdf.head()

In [ ]:
ucr_annodf=ucrdf.groupby(['Year','ORI','Population','NAME'])[allcrimes].sum().reset_index()

In [ ]:
for crime in crime_labels.keys():
    ucr_annodf[crime]=ucr_annodf[crime_labels[crime]].sum(axis=1)


In [ ]:

ucrdata=ucr_annodf[['Year','ORI','NAME','Population','Murder', 'Manslaughter', 'Rape', 
                    'Robbery', 'Assault', 'Burglary','Larceny', 'Motor Vehicle Theft']]

In [ ]:
ucrdata.head()

In [ ]:
dftoplot=ucrdata[ucrdata.ORI=='CA01942']
dftoplot.head()

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
plt.plot(dftoplot.Year,dftoplot.Murder)
ax.set_ylabel(r"{}".format('Murder'))
ax.set_xlabel(r"Year")
plt.show()

# Match data with agency/counties/city/state information

## Agency/County/State info

In [ ]:
def read_lines(filepath):
    file = open(filepath)
    contents=file.read()
    lines=contents.split('\n')
    return lines

def read_ORI_county_data(path):
    head_str='CITY/AGENCY                                               ORI7      ORI9       '
    content=read_lines(path)
    state=path.split('/')[-1].split('.')[0]
    data=[]
    for i in range(0,len(content)):
        line=content[i]
#         print()
#         print(line)
        if line=='':
            continue
        if line[-1]==":":
            county,_=line.strip(":").split(',')
            #print(county.split(' (')[1])
            countyfps=county.split(' (FIPS')[1].split(' ')[0].split('=')[1]
            county=county.split(' (')[0]
#             countyfips=countyfips.split(' ')[0].split('=')[1]
#             county=county[0]
        elif content[i]==head_str:
            continue
        else:
            elements=line.split('  ')
            elements=list(filter(lambda x: x != '', elements))
            data.append([elements[-2],countyfps,elements[0],county,state])

            
    df=pd.DataFrame(data,columns=['ORI','CountyFIPS','Agency','County','State'])
    df.ORI=df.ORI.str.strip()
    df.County=df.County.str.strip()
    df.State=df.State.str.strip()
    df.Agency=df.Agency.str.strip()

    return df

In [ ]:
path=glob.glob('../data/agencies_county_map/*.dat')[9]
print(path)
content=read_lines(path)
head_str='CITY/AGENCY                                               ORI7      ORI9       '
data=[]
state=path.split('/')[-1].split('.')[0]

for i in range(0,len(content)):
    line=content[i]
#         print()
#         print(line)
    if line=='':
        continue
    if line[-1]==":":
        county,_=line.strip(":").split(',')
        try:
            countyfps=county.split(' (FIPS')[1].split(' ')[0].split('=')[1]
        except:
            print(county,county.split(' (')[1])
        county=county.split(' (')[0]
    elif content[i]==head_str:
        continue
    else:
        elements=line.split('  ')
        elements=list(filter(lambda x: x != '', elements))
        data.append([elements[-2],countyfps,elements[0],county,state])


https://www.icpsr.umich.edu/files/NACJD/ORIs/STATESoris.html

In [ ]:
data=[]
for path in glob.glob('../data/agencies_county_map/*.dat'):
    #print(path)
    df=read_ORI_county_data(path)
    data.append(df)

df_ori_old=pd.concat(data)

In [ ]:
ucrdata.head()

In [ ]:
ucrdata[ucrdata.ORI=='MO00404']

In [ ]:
df_oritofips=pd.read_csv('../data/agencies_county_map/ori_to_fips.csv')
df_oritofips=df_oritofips[['FIPS','ORI7','ORI9','FIPS_COUNTY','FIPS_ST','NAME']]
df_oritofips.head().T

In [ ]:
ucrdata['NAME']=ucrdata['NAME'].str.replace('00000000000000000 0000000000000000000000000000 000000000000000000000000','')

In [ ]:
np.unique(ucrdata['NAME'])

In [ ]:
dict_oritofips=dict(np.transpose([df_oritofips['ORI7'],df_oritofips['FIPS'].astype(str)]))

In [ ]:
ucrdata['FIPS']=ucrdata['ORI'].map(dict_oritofips)
ucrdata['FIPS']=ucrdata['FIPS'].astype(str)


In [ ]:
ucrdata.head()

In [ ]:
round(1-len(np.unique(ucrdata[ucrdata['FIPS']=='nan'].ORI))/len(np.unique(ucrdata.ORI)),2)

In [ ]:
df_sample = pd.read_csv('../data/laucnty16.csv',index_col=0)
df_sample['State FIPS Code'] = df_sample['State FIPS Code'].apply(lambda x: str(x).zfill(2))
df_sample['County FIPS Code'] = df_sample['County FIPS Code'].apply(lambda x: str(x).zfill(3))
df_sample['FIPS'] = df_sample['State FIPS Code'] + df_sample['County FIPS Code']
df_sample['State']=[item[-1] for item in df_sample['County Name/State Abbreviation'].str.split(', ')]
df_sample['County']=[item[0] for item in df_sample['County Name/State Abbreviation'].str.split(', ')]
df_sample=df_sample[['State FIPS Code','County FIPS Code','County','State','FIPS']]
df_sample.head()

## Merge 

In [ ]:
data=pd.merge(df_sample[['County','State','FIPS']],ucrdata,on='FIPS',how='outer')
data=data[(1-np.isnan(data.Year))==1]
data.Year=data.Year.astype(int)
data['County']=data['County'].str.replace(' County','')

data.head()


In [ ]:
df_ori_old=df_ori_old[df_ori_old['ORI'].isin(list(
    np.unique(data[data.County.isna()]['ORI'].values)))][['ORI','County','State']]
df_ori_old['County']=df_ori_old['County'].str.title()
df_ori_old['County']=df_ori_old['County'].str.replace(' Municipality','')
df_ori_old[["ORI",'County','State']].head()

In [ ]:
dict_county=dict(np.transpose([df_ori_old['ORI'],df_ori_old['County'].astype(str)]))
values=data[data.County.isna()].ORI.map(dict_county)

for i,v in zip(values.index,values):
    data.at[i,'County']=v

In [ ]:
df_ori_old=df_ori_old[df_ori_old['ORI'].isin(list(
    np.unique(data[data.State.isna()]['ORI'].values)))][['ORI','County','State']]

dict_state=dict(np.transpose([df_ori_old['ORI'],df_ori_old['State'].astype(str)]))
values=data[data.State.isna()].ORI.map(dict_state)

for i,v in zip(values.index,values):
    data.at[i,'State']=v

In [ ]:
df_sample['County']=df_sample['County'].str.replace(' County','')

In [ ]:
df_sample[df_sample['County'].str.contains('Silver Bow')]

In [ ]:
for index in data[(data.ORI=='MT04700')].index:
    data.at[index,'FIPS']='30093'

In [ ]:
data['State']=data['State'].str.replace('District of Columbia','DC')

In [ ]:
data[(data.ORI=='MO00404')&(data.Year==1984)]

In [ ]:
data[(data.ORI=='ILCPD00')&(data.Year==1984)]

In [ ]:
data.to_csv('../data/clean_dataframes/crime.csv')

In [ ]:
data['County']

In [ ]:
county_data=data[(data.County=='Miami-dade')&( data.State=='FL')]
county_data.head()

fig, ax = plt.subplots(figsize=(xs,ys))
plt.plot(county_data.groupby('Year').sum()[['Murder']].reset_index().Murder)
ax.axhline(0,linestyle='dashed',color='k',linewidth=5)
ax.set_ylabel(r"{}".format('Murder'))
ax.set_xlabel(r"Year")
plt.show()

In [ ]:
data[data.State=='NB']=data[data.State=='NB'].replace('NB','NE')

In [ ]:
data[data.State=='NE']